In [1]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), ".."))
from utils import find_nearest_gene_atac_peaks, preprocessing_rna, preprocessing_atac, preprocessing_prot
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from tqdm import tqdm
tqdm.pandas()

Dataset has been downloaded from:
https://openproblems.bio/datasets/openproblems_neurips2021/bmmc_multiome/

In [2]:
adata = sc.read_h5ad("./data/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad")
adata

AnnData object with n_obs × n_vars = 69249 × 129921
    obs: 'GEX_pct_counts_mt', 'GEX_n_counts', 'GEX_n_genes', 'GEX_size_factors', 'GEX_phase', 'ATAC_nCount_peaks', 'ATAC_atac_fragments', 'ATAC_reads_in_peaks_frac', 'ATAC_blacklist_fraction', 'ATAC_nucleosome_signal', 'cell_type', 'batch', 'ATAC_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'VendorLot', 'DonorID', 'DonorAge', 'DonorBMI', 'DonorBloodType', 'DonorRace', 'Ethnicity', 'DonorGender', 'QCMeds', 'DonorSmoker'
    var: 'feature_types', 'gene_id'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts'

In [3]:
random_state = np.random.RandomState()
indices = random_state.permutation(list(range(len(adata.obs))))
adata = adata[indices,:]

# RNA layer preprocessing

In [4]:
rna = adata[:, adata.var["feature_types"] == "GEX"]

In [5]:
rna.X = np.array(rna.layers["counts"].todense())

In [6]:
rna.var["gene_stable_id"] = rna.var["gene_id"]

In [7]:
rna = preprocessing_rna(rna)
rna

AnnData object with n_obs × n_vars = 69141 × 13431
    obs: 'GEX_pct_counts_mt', 'GEX_n_counts', 'GEX_n_genes', 'GEX_size_factors', 'GEX_phase', 'ATAC_nCount_peaks', 'ATAC_atac_fragments', 'ATAC_reads_in_peaks_frac', 'ATAC_blacklist_fraction', 'ATAC_nucleosome_signal', 'cell_type', 'batch', 'ATAC_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'VendorLot', 'DonorID', 'DonorAge', 'DonorBMI', 'DonorBloodType', 'DonorRace', 'Ethnicity', 'DonorGender', 'QCMeds', 'DonorSmoker', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'feature_types', 'gene_id', 'gene_stable_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism', 'log1p', 'hvg', 'pca', 'neighbors', 'umap'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_

# ATAC layer preprocessing

In [8]:
atac = adata[:, adata.var["feature_types"] == "ATAC"]

In [ ]:
atac.X = np.array(atac.layers["counts"].todense())

In [ ]:
atac.var["start"] = list(atac.var.index.map(lambda x: x.split("-")[1]).astype("int"))
atac.var["end"] = list(atac.var.index.map(lambda x: x.split("-")[2]).astype("int"))
atac.var["gene_stable_id"] = list(atac.var.progress_apply(find_nearest_gene_atac_peaks, max_dist=10000, axis=1))

In [ ]:
atac = preprocessing_atac(atac)
atac

# Create MuData and save

In [ ]:
mdata = mu.MuData({"rna": rna, 
                   "atac": atac})

In [ ]:
mu.pp.intersect_obs(mdata)

In [ ]:
mdata.write("./data/neurips2021_multiome_BMMC.h5mu")

In [ ]:
mdata2 = mu.read("./data/neurips2021_multiome_BMMC.h5mu")
mdata2

In [ ]:
mdata2.mod["rna"]